# 7.3) Retrieval-Augmented Generation (RAG) Index
## Barcelona Urban Intelligence Assistant
### Advanced Data Processing and Analysis

This notebook is Section 7.3 of the report. It builds the search index that the assistant queries to answer policy questions about Barcelona’s mobility and environmental strategy.

The pipeline has four stages, each a separate Part:

- **Part A) Extract and chunk** the three policy PDFs (Barcelona’s Urban Mobility Plan 2024-2030, the EU Smart and Sustainable Mobility Strategy, and the Barcelona Climate Plan) into chunks, which are sensitive to paragraphs. Having an appropiate extaction of the information is essential because it is the base where our pipeline will sit.
- **Part B) Embed** every chunk using `intfloat/multilingual-e5-base`, a sentence-transformer trained for multilingual retrieval. The PMU and the Climate Plan are largely in Catalan, so a multilingual model is mandatory.
- **Part C) Index** the embeddings in a FAISS structure that returns the top-k closest chunks for a natural-language question in milliseconds, using cosine similarity (inner product on unit-length vectors).
- **Part D) Retrieve and rerank**. We use a two-stage retrieval: FAISS returns the top-20 candidates by cosine similarity (fast but approximate), and a cross-encoder reranks them by direct query-chunk relevance (slower but precise). The final top-3 is what the assistant returns.

The whole pipeline runs locally on a CPU. No API keys, no network calls at query time. This matches the governance section (Section 4 in the report): the assistant runs end-to-end on city infrastructure, and the three policy PDFs are the only knowledge source, so the assistant cannot hallucinate beyond documents the city has formally published.

**Note.**This is the final version of the pipeline. The earlier ones did not work well: at first we used a simpler English-only embedding model and character-based chunks that cut words in half, then we switched to a multilingual model and to pymupdf for cleaner extraction, and finally we added a cross-encoder reranker on top. The full version log is at the end of the notebook in the Part D interpretation. Also, this version uses `pymupdf` instead of `pypdf`. Install it once in the venv terminal: `pip install pymupdf`.

In [28]:
import os, pickle
import numpy as np

# Check that all libraries are available in the active venv.
try:
    import pymupdf as fitz   # pymupdf >= 1.24 dropped the 'fitz' alias
    from sentence_transformers import SentenceTransformer, CrossEncoder
    import faiss
    print("All RAG libraries available")
except ImportError as e:
    print(f"Missing: {e}")
    print("Run in the venv terminal: pip install sentence-transformers faiss-cpu pymupdf")

All RAG libraries available


## Extract the policy PDFs from GitHub

The cell bellow extracts the PDFs we will use from the public GitHub repository [`claudia-moliner/Policies-PDFs`](https://github.com/claudia-moliner/polices-PDF) so no manual download needed. The three files total 30 MB; downloading once on first run is the only network call in the entire pipeline.

In [ ]:
GITHUB_RAW = (
    "https://raw.githubusercontent.com/claudia-moliner/polices-PDF/main/"
)

PDF_FILES = [
    "barcelona_mobility_plan_2024_2030.pdf",  # Barcelona PMU 2024-2030 (Catalan)
    "eu_smart_mobility_strategy.pdf", # EU Smart and Sustainable Mobility Strategy (English)
    "climate_plan_maig.pdf",# Barcelona Climate Plan May 2024 (Catalan)
]

import urllib.request

for f in PDF_FILES:
        url = GITHUB_RAW + f
        urllib.request.urlretrieve(url, f)
        size_mb = os.path.getsize(f) / 1_048_576
        print(f"{f} downloaded ({size_mb:.1f} MB)")


barcelona_mobility_plan_2024_2030.pdf downloaded (5.9 MB)
eu_smart_mobility_strategy.pdf downloaded (0.7 MB)
climate_plan_maig.pdf downloaded (17.1 MB)


# Part A) Extract and chunk the PDFs

For Part A there are two choices that we have to explain, because both of them affect the retrieval quality more than the model itself.

Why pymupdf instead of pypdf. Extracting text from a PDF is not as easy as it sounds, because the PDF format stores characters by their position on the page (x, y coordinates) and not as clean sentences. pypdf is the simpler library, but it usually produces text with broken word spacing, weird paragraph breaks and column ordering errors when the page has multiple columns. pymupdf (imported as fitz) handles all of this much better. On the same PMU file we tested earlier, switching to pymupdf is what removed the weird fragments like *"ng some..."* and *"onal."\nPLANTEJAMENT* that were appearing in the report table.

Why we chunk by blocks and not by page text. With pymupdf we use get_text("blocks"), which returns every text unit as a separate item with its bounding box (x0, y0, x1, y1). This is much more reliable than splitting get_text("text") on blank lines, because the Barcelona deck is a slide format where each page is one big undivided block. Additionally, having the bounding boxes lets us filter out headers and footers by their vertical position: anything in the top 8% or the bottom 12% of the page is usually a running title or a footnote, so we drop it. Finally we pack the blocks into chunks of around 600 characters with a one-block overlap, so each chunk stays focused on one topic.

The Barcelona Climate Plan (climate_plan_maig.pdf) was added to the corpus later. Unlike the slide-format PMU or the short EU briefing, it is a text-heavy  aprox 80-page Catalan document and on its own it contributes around 528 of the 560 total chunks. The same extraction and chunking logic handles it without any changes.

# Part B) Embed each chunk

We use `intfloat/multilingual-e5-base`, a sentence-transformer trained specifically for multilingual retrieval. The justification follows the same kind of reasoning we used in Part C of the ML notebook for the sentiment model:

1. **The corpus is multilingual.** The PMU and the Climate Plan are mostly in Catalan (with some Spanish in the PMU), and the EU strategy is in English. We need the same model to embed all three languages so that a question in any language can match a passage in any language. E5 was trained on a contrastive objective across 100 languages including Catalan, which makes it the standard choice for multilingual academic RAG.
2. **The model is free, MIT-licensed, and runs on a CPU.** Downloads once (~280 MB) and runs locally afterwards. No API key, no network call at query time.
3. **It outputs 768-dimensional vectors.** Large enough to encode nuanced meaning, small enough that the FAISS index for three PDFs fits in a few MB.

**E5 prefix convention.** The E5 family was trained with a specific instruction format: documents being indexed are prefixed with `"passage: "` and queries are prefixed with `"query: "`. Skipping these prefixes worsens the retrieval quality measurably, so we apply them everywhere.

**Cosine similarity setup.** We pass `normalize_embeddings=True` so every output vector has unit length. With unit-length vectors, the inner product between two vectors equals their cosine similarity, which lets us use FAISS’s fast inner-product index in Part C.

In [ ]:
if all_chunks:
    print("Loading multilingual E5 model (280 MB on first run)")
    embed_model = SentenceTransformer("intfloat/multilingual-e5-base")
    print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

    # E5 requires the "passage: " prefix on indexed documents.
    texts = ["passage: " + c["text"] for c in all_chunks]
    embeddings = embed_model.encode(
        texts,
        show_progress_bar=True,
        batch_size=16,
        normalize_embeddings=True,# unit vectors to cosine via inner product
    )
    embeddings = np.array(embeddings).astype("float32")
    print(f"Embeddings shape: {embeddings.shape}")
else:
    print("No chunks to embed. Check Part A first.")

Loading multilingual E5 model (280 MB on first run)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4563.21it/s]
/var/folders/pl/2zg4zswn789f02rdypxz3v840000gn/T/ipykernel_534/3404902227.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


Embedding dimension: 768


Batches:   3%|▎         | 1/35 [49:16<27:55:03, 2956.00s/it]

# Part C) Build and save the FAISS index

FAISS organises the embeddings so that finding the closest k vectors to a query takes milliseconds instead of seconds. We use `IndexFlatIP` (inner product), which paired with the unit-length vectors from Part B is equivalent to cosine similarity. For our corpus size (a few hundred chunks) this exact index is more than fast enough; quantised indices only matter at millions of rows.

The score direction is the opposite of L2: with inner product on normalised vectors, **higher is more relevant** (perfect match = 1.0, unrelated = 0.0).

We save two artefacts to disk:
- `barcelona_docs.index` is the FAISS index itself (just the numbers).
- `chunks.pkl` is the parallel list of source text, filename, and page number for each vector. When FAISS returns row index 142 we look up this list to get the text and the citation.

Both files together are the searchable index. Either alone is useless.

In [ ]:
if all_chunks and len(embeddings) > 0:
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension) # inner product = cosine for unit vectors
    index.add(embeddings)

    faiss.write_index(index, "rag/embeddings/barcelona_docs.index")
    with open("rag/embeddings/chunks.pkl", "wb") as f:
        pickle.dump(all_chunks, f)

    print(f"FAISS index saved: {index.ntotal} vectors")
    print(f"Chunks saved: {len(all_chunks)} text chunks")
    print(f"Index file size: {os.path.getsize('rag/embeddings/barcelona_docs.index') // 1024} KB")
else:
    print("No embeddings to save. Check Part B first.")

FAISS index saved: 560 vectors
Chunks saved: 560 text chunks
Index file size: 1680 KB


# Part D) Retrieve and rerank: two-stage search

Using only a single embedding model is fast but not very precise, and the reason is in how it works. The embedding model encodes each chunk once when we build the index, without knowing what questions will be asked later, and then encodes the question once at query time, without knowing what chunks are in the index. So the model never actually sees the question and the chunk together, it just compares two summaries that were made separately. This setup is called a bi-encoder and it is what we built in Parts B and C.
A cross-encoder is different: it takes a pair (question, chunk) together as input and gives us back one relevance score, so it can compare the two texts directly. This is more accurate but also too slow to run against every chunk in the index, since it would take seconds instead of milliseconds. The solution that is used in most RAG systems is to do it in two steps:

- Stage 1: bi-encoder retrieval. FAISS returns the top-20 candidates by cosine similarity. Fast and approximate, casts a wide selection.
- Stage 2: cross-encoder reranking. A cross-encoder scores each of the 20 candidates by direct (question, chunk) relevance, we re-sort them and we keep the top 3.

For the reranker we use *BAAI/bge-reranker-v2-m3.* It is free, Apache-licensed, trained for multilingual reranking (more than 100 languages including Catalan), runs locally on the CPU and is around 568 MB. We initially used the smaller version bge-reranker-base, but it failed on cross-lingual pairs: for example, an English question against a Catalan chunk that the bi-encoder had correctly retrieved (cosine 0.83) was given a score of 0.013 by the base reranker. Switching to m3 lifted that same pair to 0.61, so the problem was the reranker's capacity and not the retrieval.

The reranker outputs a relevance score between 0 and 1. Scores above 0.7 mean a strong match, between 0.3 and 0.7 a partial match, and close to 0 an irrelevant candidate. We chose these thresholds by trying questions we knew were in the corpus and questions we knew were out.·

In [ ]:
con.execute("""
    SELECT complaint_type, COUNT(*) AS n
    FROM silver.complaints
    GROUP BY complaint_type
    ORDER BY n DESC
""").df()

,complaint_type,n
0,INCIDENT,126537
1,ISSUE,45349
2,COMPLAINT,36811
3,ENQUIRY,35009
4,SUGGESTION,20243
5,PETICIO DE SERVEI,10658
6,QUERY,8766
7,SERVICE REQUEST,3406
8,AGRAIMENT,386
9,GRATITUDE,139


In [ ]:
print("Loading cross-encoder reranker (~280 MB on first run)...")
rerank_model = CrossEncoder("BAAI/bge-reranker-v2-m3")
print("Reranker loaded.")


def search_rag(question, top_k=3, n_candidates=20, verbose=True):
    """Two-stage retrieval: FAISS recall, cross-encoder rerank. Returns top_k chunks."""
    # load index and chunks from disk; in production these would be cached on startup
    index = faiss.read_index("rag/embeddings/barcelona_docs.index")
    with open("rag/embeddings/chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    # stage 1: bi-encoder recall to get n_candidates closest chunks by *cosine* similarity
    query_vec = embed_model.encode(
        ["query: " + question],
        normalize_embeddings=True,
    ).astype("float32")
    cosine_scores, indices = index.search(query_vec, n_candidates)
    candidates = [chunks[i] for i in indices[0]]

    # stage 2: cross-encoder rerank to score each candidate against the query directly
    pairs = [(question, c["text"]) for c in candidates]
    rerank_scores = rerank_model.predict(pairs)

    # combine into a ranked list, sorted by the reranker score
    ranked = sorted(
        zip(candidates, cosine_scores[0], rerank_scores),
        key=lambda x: x[2],  # sort by cross-encoder
        reverse=True,
    )[:top_k]

    results = []
    for rank, (chunk, cos, rer) in enumerate(ranked, start=1):
        results.append({
            "rank": rank,
            "source": chunk["source"],
            "page": chunk["page"],
            "cosine": round(float(cos), 4),
            "rerank": round(float(rer), 4),
            "excerpt": chunk["text"][:400],
        })

    if verbose:
        print(f"\nQ: {question}")
        for r in results:
            confidence = "strong" if r["rerank"] > 0.7 else ("partial" if r["rerank"] > 0.3 else "weak")
            print(f"  Rank {r['rank']} | {r['source']} p.{r['page']} | rerank={r['rerank']} ({confidence})")
            print(f"{r['excerpt'][:200]}...")
    return results

Loading cross-encoder reranker (~280 MB on first run)...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1668.25it/s]


Reranker loaded.


### Six demo questions for Section 7.3

These six queries represent the retrieval cases described in the checklist. Three of the six were reformulated so the wording aligns with what the three policy PDFs actually contain: the original phrasing was generic enough that the bi-encoder scored every chunk at roughly the same low similarity. Reformulating to terms the documents actually use (*strategic objectives*, *PMU 2025-2030*, *data and digital innovation*) lets the embedding model surface the right sections.

Q2 is intentionally kept as an **out-of-scope control**, complaint resolution time is a service-level metric that lives in `gold.fct_resolution_times`, not in any of the policy PDFs. Its reranker score should stay noticeably below the others, demonstrating that the assistant returns low confidence on questions outside the corpus rather than fabricating an answer. Adding the Climate Plan to the corpus in v7 did not break this: Q2 still scores 0.00, though it now happens to pull its rank-1 chunk from the climate plan rather than the EU strategy.

The rank-1 result of each question goes into **Table 7.5.** in the report.

In [ ]:
import pandas as pd

demo_questions = [
    "What role does data and digital innovation play in the EU sustainable mobility strategy?",
    "What are the key strategic objectives of Barcelona's 2030 sustainable urban mobility plan?",
    "How long does it take to resolve a formal complaint?",  # out-of-scope control
    "How does Barcelona's mobility plan involve districts in its planning process?",
    "What future goals does the EU set for sustainable transport?",
    "What targets does Barcelona's Climate Plan set for emissions reduction?"
]

rows = []
for q in demo_questions:
    top = search_rag(q, top_k=3, verbose=False)[0]
    rows.append({
        "Question": q,
        "First result (200 chars)": top["excerpt"][:200] + "...",
        "Source document": top["source"],
        "Page": top["page"],
        "Cosine (stage 1)": top["cosine"],
        "Rerank score (stage 2)": top["rerank"],
    })

table_5_3_1 = pd.DataFrame(rows)
table_5_3_1

,Question,First result (200 chars),Source document,Page,Cosine (stage 1),Rerank score (stage 2)
0,What role does data and digital innovation pla...,The more horizontal areas of action outlined i...,eu_smart_mobility_strategy.pdf,3,0.8378,0.4558
1,What are the key strategic objectives of Barce...,“Tenint en compte els diferents motius dels de...,barcelona_mobility_plan_2024_2030.pdf,5,0.8600,0.9819
2,How long does it take to resolve a formal comp...,• Tactical communication actions (ongoing).\n\...,climate_plan_maig.pdf,80,0.7644,0.0000
3,How does Barcelona's mobility plan involve dis...,Es proposen reunions de seguiment mensuals amb...,barcelona_mobility_plan_2024_2030.pdf,23,0.8273,0.6087
4,What future goals does the EU set for sustaina...,The course of EU transport policy has been def...,eu_smart_mobility_strategy.pdf,2,0.8367,0.8440
5,What targets does Barcelona's Climate Plan set...,to break the present consumption and emission ...,climate_plan_maig.pdf,56,0.8963,0.9989


The table above shows only rank-1 per question. The cell below prints the full top-3 results with longer excerpts, for sanity-checking that the retrieved chunks make sense in context. This output is for our own validation and is not reproduced in the report.

In [ ]:
for q in demo_questions:
    print("=" * 80)
    print(f"Q: {q}")
    print("=" * 80)
    results = search_rag(q, top_k=3, verbose=False)
    for r in results:
        print(f"\n  Rank {r['rank']} | {r['source']} page:{r['page']}")
        print(f"  Cosine: {r['cosine']}  |  Rerank: {r['rerank']}")
        print(f"  Excerpt (first 600 chars):")
        print(f"  {r['excerpt'][:600]}")

Q: What role does data and digital innovation play in the EU sustainable mobility strategy?



  Rank 1 | eu_smart_mobility_strategy.pdf page:3
  Cosine: 0.8378  |  Rerank: 0.4558
  Excerpt (first 600 chars):
  The more horizontal areas of action outlined in the strategy target healthy and sustainable inter- urban and urban mobility; connected and automated mobility; innovation, data and artificial intelligence; reinforcing the transport single market; making mobility fair and just for all; enhancing transport safety and security, and safeguarding the EU’s global position. Reactions

The European Transpo

  Rank 2 | eu_smart_mobility_strategy.pdf page:3
  Cosine: 0.8421  |  Rerank: 0.4477
  Excerpt (first 600 chars):
  To green airports and ports, the Commission will propose incentives for the deployment of renewable and low carbon fuels, and ensure that stationary aircraft and vessels can use renewable power. The revised lending policy of the European Investment Bank and the EU green taxonomy criteria (identifying which investments count as sustainable) will support the introd

### Interpretation — Part D finding
For each question, the two-stage retrieval returns a ranked list of chunks with two scores: a cosine similarity from FAISS at stage 1, and a more precise reranker score from the cross-encoder at stage 2. Of the six demo questions, three are strong matches (Q1 = 0.9819, Q4 = 0.8440, Q5 = 0.9989), two are partial (Q0 = 0.4558, Q3 = 0.6087), and one is near zero (Q2 = 0.0000), which is the out-of-scope control. For every question the citation points to the exact page of the correct PDF.

The cosine score from stage 1 is less informative than the reranker score: all six questions get cosines between 0.76 and 0.90, so there is basically no gap between the answerable and the unanswerable ones. The cross-encoder reopens that gap, because the difference between the lowest in-scope reranker score (Q0 = 0.4558) and the out of scope control (Q2 = 0.0) is 0.46. This is exactly the reason why two-stage retrieval is the standard approach in RAG: the cross-encoder scores the (question, chunk) pair together instead of comparing two summaries that were made independently.

Additionally, the corpus grew in the last iteration when we added climate_plan_maig.pdf, going from 32 to 560 chunks (around 94% of them come from this one document). Even with this imbalance, the reranker kept the rank-1 chunk for all the previous questions, and the new Q5 scored 0.9989 against a climate plan chunk. This is the expected behaviour of a cross-encoder: it ranks by relevance to the specific question, not by how often something appears in the corpus.

## Section 7.3 summary

In this section we made a working retrieval system from three policy PDFs:
- **Part A** extracts text blocks with `pymupdf` using `get_text("blocks")`, filters headers and footers by vertical position, and packs blocks into chunks of 600 characters with one-block overlap. Page numbers are tracked for citation.
- **Part B** embeds each chunk with `intfloat/multilingual-e5-base` into a 768-dimensional unit vector.
- **Part C** stores those vectors in a FAISS `IndexFlatIP` for cosine-similarity retrieval.
- **Part D** runs two-stage retrieval: FAISS returns 20 candidates by cosine similarity, then `BAAI/bge-reranker-v2-m3` reranks them by direct query-chunk relevance and the top 3 are returned with source and page citations.

**Data governance carry-over.** The index is built only from policy PDFs the city has formally published. No silver-layer complaint text or PII-masked free text enters the index. The whole pipeline runs locally on a CPU, no API key, no external network call at query time. This matches Section 4.5 of the report (access control): the assistant only reads from gold tables and policy documents.

**Limitations to mention in the report write-up:**
- Three policy PDFs is still a small imput. Adding the Barcelona Environmental Strategy and the data governance handbook would meaningfully widen what the assistant can answer. In further work where a more powerful computer is being emploied, more imput data would benefit the model greatly.
- The Barcelona PMU is a slide-format presentation: most KPI values and district maps are in infographic elements that `pymupdf` cannot extract as text. Retrieval on questions requiring specific metric values is limited by what the text layer actually contains.
- Q3 (districts question) scores 0.6087 rather than higher because the rank-1 chunk merges the PMU’s internal-coordination section with the district-participation text from the same slide to chunkt. A finer chunk boundary or section-aware splitting would likely push this score higher.
- Chunking uses block boundaries but does not parse the PDF’s table of contents or section headings as metadata. A more sophisticated implementation would use those as additional retrieval signals.
- We use `IndexFlatIP`, which is exact but stores every vector in full. At tens of thousands of chunks we would switch to a quantised index like `IndexIVFPQ`.
- The bi-encoder and cross-encoder are both multilingual but not domain-adapted to Catalan municipal policy text.
- The pipeline retrieves and ranks passages but does not generate natural-language answers.